# 🎯 Multi-Camera Person Re-Identification System (Automated)

This notebook is fully automated to run the **v3 Identity Stability Engine**.

**Instructions:**
1. Make sure `cctv1.mp4` and `cctv2.mp4` are in the main folder of your Google Drive.
2. Go to the top menu and click **Runtime > Run all**.
3. It will automatically ask for Drive permission, process the videos, and save the final outputs directly back to your Google Drive!

In [ ]:
# 1. Check GPU
import torch
print(f"\nPyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Install Dependencies
!pip install -q ultralytics faiss-cpu scipy opencv-python pyyaml lap Cython gdown tensorboard h5py
!pip install -q git+https://github.com/KaiyangZhou/deep-person-reid.git
print("\n✅ All dependencies installed!")

In [ ]:
# 3. Clone Repository & Setup
import os
!rm -rf /content/person-reid
!git clone https://github.com/hypertonny/person-reid.git /content/person-reid

os.chdir('/content/person-reid/multi_camera_reid')
for d in ['data/clips', 'models', 'outputs/annotated', 'outputs/gallery_state', 'logs']:
    os.makedirs(d, exist_ok=True)
print("\n✅ Repository Cloned & Folders Created!")

In [ ]:
# 4. Mount Google Drive & Copy Videos
from google.colab import drive
drive.mount('/content/drive')

print("\nCopying videos from Google Drive...")
!cp "/content/drive/MyDrive/cctv1.mp4" data/clips/cctv1.mp4
!cp "/content/drive/MyDrive/cctv2.mp4" data/clips/cctv2.mp4

if os.path.exists("data/clips/cctv1.mp4") and os.path.exists("data/clips/cctv2.mp4"):
    print("✅ Videos successfully copied from Drive!")
else:
    print("❌ ERROR: Videos not found in your Drive! Please make sure they are named cctv1.mp4 and cctv2.mp4 and are in your main Drive folder.")

In [ ]:
# 5. Run the Multi-Camera ReID Pipeline
print("\n🚀 Starting the AI Engine...")
!python orchestrator.py --config config.yaml

In [ ]:
# 6. Convert & Save Videos back to Google Drive
import subprocess

print("\n🎬 Converting and saving final videos to Google Drive...")
for cam in ['cctv-1', 'cctv-2']:
    src = f'outputs/annotated/{cam}_annotated.mp4'
    dst = f'/content/drive/MyDrive/{cam}_reid_final.mp4'
    if os.path.exists(src):
        subprocess.run(['ffmpeg', '-y', '-i', src, '-vcodec', 'libx264', '-acodec', 'aac', dst], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"✅ Saved {cam} to Drive as {cam}_reid_final.mp4")
    else:
        print(f"⚠️ Could not find output for {cam}")

print("\n🎉 ALL DONE! You can now view the videos in your Google Drive.")